# 01 · Data exploration

A look at the PTB-XL metadata, the raw 12-lead signals and the diagnostic labels before any modelling.

**Dataset:** [PTB-XL](https://physionet.org/content/ptb-xl/): 21,799 clinical 12-lead ECGs (10 s each) from 18,869 patients, annotated by cardiologists with SCP-ECG statements.

All loading and preprocessing logic lives in `ptbxl/data.py`; this notebook only calls into it and visualises the results.
To produce the cached training data, run `python prepare_data.py` from the repo root.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))  # make the `ptbxl` package importable

DATA_DIR = os.environ.get("PTBXL_DATA_DIR", "../data/ptb-xl")
NPZ_PATH = os.environ.get("PTBXL_NPZ", "../data/processed/data.npz")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno

from ptbxl.data import load_metadata, load_raw_data, build_labels, build_tabular_features

## 1. Metadata

`ptbxl_database.csv` has one row per ECG: patient demographics, recording info, signal-quality notes, the SCP codes (as a `{code: likelihood}` dict) and the pre-defined stratified fold `strat_fold`.
`scp_statements.csv` describes every SCP code; only the **diagnostic** statements (44 of them) are kept.

In [ ]:
ECG_df, SCP = load_metadata(DATA_DIR)
print(f"{len(ECG_df)} records, {ECG_df.patient_id.nunique()} patients, {len(SCP)} diagnostic statements")
ECG_df.head()

## 2. Missing values

Height and weight are missing for most records, and several signal-quality columns are only filled when a problem was observed, so "missing" there really means "no issue".

In [ ]:
print(ECG_df.isnull().sum())
msno.matrix(ECG_df, figsize=(20, 8))
plt.show()

In [ ]:
ECG_df[[c for c in ECG_df.columns if c != "scp_codes"]].nunique(dropna=True)

## 3. Raw signals

Each record is a `(1000, 12)` array at 100 Hz (10 s × 12 leads). Below is the first record, one lead per row.

In [ ]:
signals = load_raw_data(ECG_df.iloc[:5], 100, DATA_DIR)
sample = signals[0]
lead_names = ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"]

fig, axes = plt.subplots(12, 1, figsize=(20, 12), sharex=True)
for i, ax in enumerate(axes):
    ax.plot(sample[:, i], linewidth=0.8)
    ax.set_ylabel(lead_names[i], rotation=0, labelpad=20)
axes[-1].set_xlabel("sample (100 Hz)")
plt.tight_layout()
plt.show()

## 4. Labels

The target is a **multi-hot vector over the 44 diagnostic statements**. A record can carry several diagnoses at once, so this is multi-label classification (an independent sigmoid per label), not multi-class.

In [ ]:
Z, label_names = build_labels(ECG_df, SCP)
Z_df = pd.DataFrame(Z, index=ECG_df.index, columns=label_names)

n_labels = Z.sum(axis=1)
print(f"Records with >1 label : {(n_labels > 1).sum()}")
print(f"Records with 0 labels : {(n_labels == 0).sum()}")
print(f"Mean labels / record  : {n_labels.mean():.2f}")
print(f"Fraction of positive cells in Z: {Z.mean():.3%}")

The last number explains the evaluation choices: only about 3% of the label matrix is positive. A model that predicts **all zeros** would already get about 97% "label-wise accuracy", which is why that metric is not used to judge the model (see notebook 02).

In [ ]:
counts = Z_df.sum().sort_values(ascending=False)
plt.figure(figsize=(14, 5))
sns.barplot(x=counts.index, y=counts.values, color="steelblue")
plt.yscale("log")
plt.xticks(rotation=90)
plt.ylabel("# records (log scale)")
plt.title("Diagnostic label frequency")
plt.tight_layout()
plt.show()

The labels are heavily imbalanced: `NORM` appears in thousands of records while several statements have only a handful of examples.

## 5. Tabular features

Alongside the signal, the model gets 8 metadata features (age, sex, height, weight, infarction stadium 1/2, pacemaker, extra beats). Missing values are filled as designed in the original notebook: 0 for age and weight, the median of valid heights for height.

In [ ]:
X = build_tabular_features(ECG_df)
X.describe().T

## 6. Train / validation / test split

PTB-XL ships with 10 stratified folds in which **all records of a patient are in the same fold**, so there is no patient leakage between splits. Following the dataset authors' recommendation: folds 1–8 train, fold 9 validation, fold 10 test.

In [ ]:
split = pd.cut(ECG_df.strat_fold, bins=[0, 8, 9, 10], labels=["train", "valid", "test"])
split.value_counts().reindex(["train", "valid", "test"])